In [28]:
import pandas as pd
import numpy as np
from scipy.interpolate import LinearNDInterpolator
from sklearn.model_selection import train_test_split
import os

PROJECT_ROOT = r'C:\Users\user\moscow_apartments'
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'apartments_features.csv'))

baseline_features = [
    'Minutes to metro', 'Number of rooms', 'Area', 'Kitchen area',
    'Floor', 'Number of floors', 'floor_ratio', 'is_first_floor',
    'is_last_floor', 'kitchen_area_ratio', 'living_area_ratio',
    'region_encoded', 'apartment_type_encoded', 'renovation_encoded'
]

X = df[baseline_features]
y = df['log_price']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape,X_test.shape)

(14472, 14) (3619, 14)


In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error

model = LinearRegression()
model.fit(X_train,y_train)

y_pred_log = model.predict(X_test)

y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

mae = mean_absolute_error(y_test_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
mape = np.mean(np.abs((y_test_actual - y_pred) / y_test_actual)) * 100

print(f"MAE: {mae:,.0f} руб")
print(f"RMSE: {rmse:,.0f} руб")
print(f"MAPE: {mape:.2f}%")

MAE: 10,105,116 руб
RMSE: 32,887,030 руб
MAPE: 22.62%


In [30]:
from catboost import CatBoostRegressor

catboost_features = [
    'Minutes to metro', 'Number of rooms', 'Area', 'Kitchen area',
    'Floor', 'Number of floors', 'floor_ratio', 'is_first_floor',
    'is_last_floor', 'kitchen_area_ratio', 'living_area_ratio',
    'metro_station', 'Region', 'Apartment type', 'Renovation'
]


X_cb = df[catboost_features]
X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(X_cb, y, test_size=0.2, random_state=42)

cat_features = ['metro_station', 'Region', 'Apartment type', 'Renovation']

cb_model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=100, random_state=42)
cb_model.fit(X_train_cb, y_train_cb, cat_features=cat_features)

y_pred_cb_log = cb_model.predict(X_test_cb)
y_pred_cb = np.expm1(y_pred_cb_log)

mae_cb = mean_absolute_error(y_test_actual, y_pred_cb)
rmse_cb = np.sqrt(mean_squared_error(y_test_actual, y_pred_cb))
mape_cb = np.mean(np.abs((y_test_actual - y_pred_cb) / y_test_actual)) * 100

print(f"CatBoost MAE: {mae_cb:,.0f} руб")
print(f"CatBoost RMSE: {rmse_cb:,.0f} руб")
print(f"CatBoost MAPE: {mape_cb:.2f}%")

0:	learn: 0.9383135	total: 18.6ms	remaining: 9.28s
100:	learn: 0.2308458	total: 2.4s	remaining: 9.48s
200:	learn: 0.2141774	total: 4.44s	remaining: 6.6s
300:	learn: 0.2034293	total: 6.39s	remaining: 4.23s
400:	learn: 0.1943101	total: 8.36s	remaining: 2.06s
499:	learn: 0.1871402	total: 10.3s	remaining: 0us
CatBoost MAE: 6,260,836 руб
CatBoost RMSE: 19,431,126 руб
CatBoost MAPE: 13.47%


In [31]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'CatBoost'],
    'MAE': [mae, mae_cb],
    'RMSE': [rmse, rmse_cb],
    'MAPE (%)': [mape, mape_cb]
})
print(comparison)

# Feature importance CatBoost
importance = pd.DataFrame({
    'feature': X_train_cb.columns,
    'importance': cb_model.get_feature_importance()
}).sort_values('importance', ascending=False)
print(importance)

               Model           MAE          RMSE   MAPE (%)
0  Linear Regression  1.010512e+07  3.288703e+07  22.620354
1           CatBoost  6.260836e+06  1.943113e+07  13.469298
               feature  importance
2                 Area   47.047922
13      Apartment type   11.054436
14          Renovation    9.858825
12              Region    9.029649
11       metro_station    6.349972
5     Number of floors    5.286503
0     Minutes to metro    2.932259
1      Number of rooms    2.428076
3         Kitchen area    1.866802
10   living_area_ratio    1.713903
9   kitchen_area_ratio    1.186773
6          floor_ratio    0.677259
4                Floor    0.427952
7       is_first_floor    0.103601
8        is_last_floor    0.036069


In [32]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mape_scores = []
for train_idx, val_idx in kf.split(X_cb):
    X_tr, X_val = X_cb.iloc[train_idx], X_cb.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_state=42)
    model.fit(X_tr, y_tr, cat_features=cat_features)

    pred = np.expm1(model.predict(X_val))
    actual = np.expm1(y_val)
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    mape_scores.append(mape)

print(f"MAPE по фолдам: {mape_scores}")
print(f"Среднее MAPE: {np.mean(mape_scores):.2f}% (±{np.std(mape_scores):.2f})")

MAPE по фолдам: [np.float64(13.401219148827492), np.float64(13.209756297333856), np.float64(13.484751560708034), np.float64(13.7557877827619), np.float64(14.087470583634373)]
Среднее MAPE: 13.59% (±0.31)


In [33]:
from catboost import CatBoostRegressor
import numpy as np

model = CatBoostRegressor(
    cat_features=cat_features,
    loss_function='RMSE',
    verbose=0,
    random_state=42
)

param_grid = {
    'iterations': [300, 500, 700],
    'learning_rate': [0.03, 0.05, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [3, 5, 7]
}

grid_search_result = model.grid_search(
    param_grid,
    X=X_train_cb,
    y=y_train_cb,
    cv=3,
    verbose=False,
    plot=False
)

print(f"Лучшие параметры: {grid_search_result['params']}")

best_params = grid_search_result['params']

final_model = CatBoostRegressor(
    **best_params,
    cat_features=cat_features,
    loss_function='RMSE',
    verbose=100,
    random_state=42
)

final_model.fit(X_train_cb, y_train_cb)

y_pred_final = np.expm1(final_model.predict(X_test_cb))
mape_final = np.mean(np.abs((y_test_actual - y_pred_final) / y_test_actual)) * 100

print(f"\nMAPE после тюнинга: {mape_final:.2f}%")
print(f"MAPE до тюнинга (по кросс-валидации): 13.59%")


bestTest = 0.2412794064
bestIteration = 299


bestTest = 0.2291264045
bestIteration = 299


bestTest = 0.2183942045
bestIteration = 299


bestTest = 0.2443453793
bestIteration = 299


bestTest = 0.2321204418
bestIteration = 299


bestTest = 0.2204958581
bestIteration = 299


bestTest = 0.2474231298
bestIteration = 299


bestTest = 0.2337855463
bestIteration = 299


bestTest = 0.2224035522
bestIteration = 299


bestTest = 0.2271149189
bestIteration = 499


bestTest = 0.2192207204
bestIteration = 499


bestTest = 0.2095450273
bestIteration = 499


bestTest = 0.2293840039
bestIteration = 499


bestTest = 0.2217217677
bestIteration = 499


bestTest = 0.2108348108
bestIteration = 498


bestTest = 0.231336387
bestIteration = 498


bestTest = 0.2232529588
bestIteration = 499


bestTest = 0.2126193612
bestIteration = 499


bestTest = 0.2211982181
bestIteration = 697


bestTest = 0.2131309619
bestIteration = 699


bestTest = 0.2043230534
bestIteration = 699


bestTest = 0.22385693
bestIteratio

In [34]:
y_pred_train = np.expm1(final_model.predict(X_train_cb))
y_train_actual = np.expm1(y_train_cb)
mape_train = np.mean(np.abs((y_train_actual - y_pred_train) / y_train_actual)) * 100

print(f"Train MAPE: {mape_train:.2f}%")
print(f"Test MAPE: {mape_final:.2f}%")

Train MAPE: 10.31%
Test MAPE: 12.33%


In [40]:
models_dir = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(models_dir, exist_ok=True)
final_model.save_model(os.path.join(PROJECT_ROOT, 'models', 'catboost_final.cbm'))